In [182]:
import os
import dotenv
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

## Define embedding 

In [183]:
from llama_index.embeddings.openai import OpenAIEmbedding
embed_model = OpenAIEmbedding()

### LLM model

In [184]:
from llama_index.llms.openai import OpenAI
llm = OpenAI(model= "gpt-5-nano",temperature=0)

In [185]:
from llama_index.core import Settings

Settings.embed_model = embed_model
Settings.llm = llm

### Ingestion Pipeline

In [186]:
# set chunks
Settings.chunk_size = 1024

In [187]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import TokenTextSplitter

# load 
documents = SimpleDirectoryReader('data').load_data()

In [188]:
#chunking strategy
text_splitter = TokenTextSplitter()

## Vector DB

In [189]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

# create local in memory client
chroma_client = chromadb.EphemeralClient()
#create a collection 
chroma_collection = chroma_client.create_collection("ps-foo-rag", get_or_create=True)
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [190]:
# ingestion  pipeline
from llama_index.core.ingestion import IngestionPipeline
pipeline = IngestionPipeline(
    transformations=[
        text_splitter,
        embed_model
    ],
    vector_store = vector_store
)

In [191]:
nodes = pipeline.run(documents=documents)
print(f"ingested {len(nodes)} nodes")

2025-10-29 10:36:18,748 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


ingested 1 nodes


# RAG Pipeline

In [192]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex.from_vector_store(vector_store=vector_store,
                                                  embed_model=embed_model)

In [193]:
# create semantic query engine
vector_query_engine = vector_index.as_query_engine()

In [194]:
response = vector_query_engine.query("who is the ceo of the company?")
print(response)

2025-10-29 10:36:19,437 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-29 10:36:25,150 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Tony
